Machine Learning 203 Data Challenge



In [351]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder
from category_encoders import TargetEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
import lightgbm as lgb

In [210]:
x_insample = pd.read_csv(r"x_train_final.csv")
y_insample = pd.read_csv(r"y_train_final_j5KGWWK.csv")
x_outsample = pd.read_csv(r"x_test_final.csv")

In [3]:
#Try 1: Random Linear Factors Combinaison (Score: 0,933)

y_outsample = 0.6*(0.5*x_outsample["p0q2"]+0.35*x_outsample["p0q3"]+0.15*x_outsample["p0q4"])
+0.4*(0.5*x_outsample["p2q0"]+0.35*x_outsample["p3q0"]+0.15*x_outsample["p4q0"])
y_outsample = y_outsample.round(0).astype(int)
y_outsample.to_csv("y_outsample_v1.csv")

In [ ]:
#Try 2: Linear Regression (Score: 0,866)

X = x_insample[['p2q0', 'p3q0', 'p4q0', 'p0q2','p0q3','p0q4']]
Y = y_insample
model = LinearRegression().fit(X, Y)

new_X = x_outsample[['p2q0', 'p3q0', 'p4q0', 'p0q2','p0q3','p0q4']]
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)
y_pred = pd.DataFrame(y_pred)
y_pred = y_pred.iloc[:, 1:]
y_pred.to_csv("y_pred_v2.csv")

In [384]:
#Split the in sample Data to be able to test the model locally
split_idx = int(len(x_insample) * 1)
x_train, x_val = x_insample.iloc[:split_idx], x_insample.iloc[split_idx:]
y_train, y_val = y_insample.iloc[:split_idx], y_insample.iloc[split_idx:]

y_train = y_train.iloc[:, -1]
y_val = y_val.iloc[:, -1]

In [62]:
#Testing with our Linear Regression model  
X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2','p0q3','p0q4']]
Y = y_train
model = LinearRegression().fit(X, Y)

new_X = x_val[['p2q0', 'p3q0', 'p4q0', 'p0q2','p0q3','p0q4']]
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)


# Compute MAE
mae = mean_absolute_error(y_val, y_pred)
print(f"Local MAE: {mae:.4f}")

Local MAE: 0.8219


In [61]:
# Testing a Ridge Regression model  
X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']]
Y = y_train
model = Ridge(alpha=100.0).fit(X, Y)  

new_X = x_val[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']]
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)


# Compute MAE
mae = mean_absolute_error(y_val, y_pred)
print(f"Local MAE: {mae:.4f}")

Local MAE: 0.8219


In [60]:
# Testing a Lasso Regression model
X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2','p0q3','p0q4']]
Y = y_train
model = Lasso(alpha=1.0).fit(X, Y)  

new_X = x_val[['p2q0', 'p3q0', 'p4q0', 'p0q2','p0q3','p0q4']]
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)

# Compute MAE
mae = mean_absolute_error(y_val, y_pred)
print(f"Local MAE: {mae:.4f}")

Local MAE: 0.8705


In [72]:

# Trying a Decision Tree model
X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2','p0q3','p0q4']]
Y = y_train
model = DecisionTreeRegressor(max_depth=8, random_state=42)
model.fit(X, Y)

# Make predictions
new_X = x_val[['p2q0', 'p3q0', 'p4q0', 'p0q2','p0q3','p0q4']]
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)

# Compute MAE
mae = mean_absolute_error(y_val, y_pred)
print(f"Local MAE: {mae:.4f}")


Local MAE: 0.7806


In [ ]:
#Try 3: Basic Decision Tree model (Score: 0,833)

X = x_insample[['p2q0', 'p3q0', 'p4q0', 'p0q2','p0q3','p0q4']]
Y = y_insample
model = DecisionTreeRegressor(max_depth=6, random_state=42)
model.fit(X, Y)

# Make predictions
new_X = x_outsample[['p2q0', 'p3q0', 'p4q0', 'p0q2','p0q3','p0q4']]
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)
y_pred = pd.DataFrame(y_pred)
y_pred = y_pred.iloc[:, 1:]
y_pred.to_csv("y_pred_v3.csv")

In [233]:
# Trying a Random Forest model
X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']]
Y = y_train

model = RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X, Y)

# Make predictions
new_X = x_val[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']]
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)

# Compute MAE
mae = mean_absolute_error(y_val, y_pred)
print(f"Local MAE: {mae:.4f}")

Local MAE: 0.7918


In [230]:
#Try 4: Basic Random Forest model (Score: 0,793)

# Trying a Random Forest model
X = x_insample[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']]
Y = y_insample

model = RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X, Y)

# Make predictions
new_X = x_outsample[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']]
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)
y_pred = pd.DataFrame(y_pred)
y_pred = y_pred.iloc[:, 1:]
y_pred.to_csv("y_pred_v4.csv")

In [232]:
# Train a XGBoost model on raw data 

X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']]
Y = y_train

# Initialize and train the XGBoost model
model = xgb.XGBRegressor(
    n_estimators=375,   # Number of trees (can be tuned)
    max_depth=12,        # Depth of each tree
    learning_rate=0.01,  # Step size (can be tuned)
    subsample=0.9,      # Row sampling
    colsample_bytree=0.8,  # Feature sampling
    random_state=42,
    n_jobs=-1           # Use all CPU cores
    )

model.fit(X, Y)

# Make predictions
new_X = x_val[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']]
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)

# Compute MAE
mae = mean_absolute_error(y_val, y_pred)
print(f"Local MAE: {mae:.4f}")

Local MAE: 0.7856


In [ ]:
#Try 5: Basic XGBoost Model on raw data (Score: )

# Prepare data
X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']]
Y = y_train

# Initialize and train the XGBoost model
model = xgb.XGBRegressor(
    n_estimators=375,   # Number of trees (can be tuned)
    max_depth=12,        # Depth of each tree
    learning_rate=0.01,  # Step size (can be tuned)
    subsample=0.9,      # Row sampling
    colsample_bytree=0.8,  # Feature sampling
    random_state=42,
    n_jobs=-1           # Use all CPU cores
    )

model.fit(X, Y)

# Make predictions
new_X = x_outsample[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']]
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)
y_pred = pd.DataFrame(y_pred)
y_pred.to_csv("y_pred_v5.csv")

In [234]:
#Feature Engineering v1

# Copy dataset
X = x_train.copy()
new_X = x_val.copy()

# Add rolling mean (3-day & 7-day window)
X["rolling_mean_3d"] = X[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].mean(axis=1).rolling(window=3, min_periods=1).mean()
X["rolling_mean_7d"] = X[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].mean(axis=1).rolling(window=7, min_periods=1).mean()

# Add rolling std (3-day & 7-day window)
X["rolling_std_3d"] = X[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].std(axis=1).rolling(window=3, min_periods=1).std()
X["rolling_std_7d"] = X[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].std(axis=1).rolling(window=7, min_periods=1).std()

# Add Rate of Change Features (handling NaNs)
for col in ['p2q0', 'p3q0', 'p4q0']:
    X[f"roc_{col}_3d"] = X[col].pct_change(periods=3).fillna(0).replace([np.inf, -np.inf], 0)
    X[f"roc_{col}_7d"] = X[col].pct_change(periods=7).fillna(0).replace([np.inf, -np.inf], 0)

# Apply same transformation to validation set
new_X["rolling_mean_3d"] = new_X[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].mean(axis=1).rolling(window=3, min_periods=1).mean()
new_X["rolling_mean_7d"] = new_X[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].mean(axis=1).rolling(window=7, min_periods=1).mean()

new_X["rolling_std_3d"] = new_X[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].std(axis=1).rolling(window=3, min_periods=1).std()
new_X["rolling_std_7d"] = new_X[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].std(axis=1).rolling(window=7, min_periods=1).std()

for col in ['p2q0', 'p3q0', 'p4q0']:
    new_X[f"roc_{col}_3d"] = new_X[col].pct_change(periods=3).fillna(0).replace([np.inf, -np.inf], 0)
    new_X[f"roc_{col}_7d"] = new_X[col].pct_change(periods=7).fillna(0).replace([np.inf, -np.inf], 0)

# Define features
features = ['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4',
            'rolling_mean_3d', 'rolling_mean_7d', 'rolling_std_3d', 'rolling_std_7d',
            'roc_p2q0_3d', 'roc_p3q0_3d', 'roc_p4q0_3d',
            'roc_p2q0_7d', 'roc_p3q0_7d', 'roc_p4q0_7d']

# Ensure no missing values
X_train_fe = X[features].fillna(0)
X_val_fe = new_X[features].fillna(0)

# Train XGBoost
model = xgb.XGBRegressor(n_estimators=380, learning_rate=0.01, max_depth=12, 
                         subsample=0.9, colsample_bytree=0.8, random_state=42)

model.fit(X_train_fe, y_train)

# Make predictions
y_pred = model.predict(X_val_fe)
y_pred = y_pred.round(0).astype(int)

# Compute MAE
mae = mean_absolute_error(y_val, y_pred)
print(f"New MAE after adding rate of change: {mae:.4f}")

New MAE after adding rate of change: 0.7735


In [ ]:
# Try 6: XGBoost Model with rolling statistics (Score: 0,801)

# Copy dataset
X = x_train.copy()
new_X = x_outsample.copy()

# Add rolling mean (3-day & 7-day window)
X["rolling_mean_3d"] = X[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].mean(axis=1).rolling(window=3, min_periods=1).mean()
X["rolling_mean_7d"] = X[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].mean(axis=1).rolling(window=7, min_periods=1).mean()

# Add rolling std (3-day & 7-day window)
X["rolling_std_3d"] = X[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].std(axis=1).rolling(window=3, min_periods=1).std()
X["rolling_std_7d"] = X[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].std(axis=1).rolling(window=7, min_periods=1).std()

# Add Rate of Change Features (handling NaNs)
for col in ['p2q0', 'p3q0', 'p4q0']:
    X[f"roc_{col}_3d"] = X[col].pct_change(periods=3).fillna(0).replace([np.inf, -np.inf], 0)
    X[f"roc_{col}_7d"] = X[col].pct_change(periods=7).fillna(0).replace([np.inf, -np.inf], 0)

# Apply same transformation to validation set
new_X["rolling_mean_3d"] = new_X[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].mean(axis=1).rolling(window=3, min_periods=1).mean()
new_X["rolling_mean_7d"] = new_X[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].mean(axis=1).rolling(window=7, min_periods=1).mean()

new_X["rolling_std_3d"] = new_X[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].std(axis=1).rolling(window=3, min_periods=1).std()
new_X["rolling_std_7d"] = new_X[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].std(axis=1).rolling(window=7, min_periods=1).std()

for col in ['p2q0', 'p3q0', 'p4q0']:
    new_X[f"roc_{col}_3d"] = new_X[col].pct_change(periods=3).fillna(0).replace([np.inf, -np.inf], 0)
    new_X[f"roc_{col}_7d"] = new_X[col].pct_change(periods=7).fillna(0).replace([np.inf, -np.inf], 0)

# Define features
features = ['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4',
            'rolling_mean_3d', 'rolling_mean_7d', 'rolling_std_3d', 'rolling_std_7d',
            'roc_p2q0_3d', 'roc_p3q0_3d', 'roc_p4q0_3d',
            'roc_p2q0_7d', 'roc_p3q0_7d', 'roc_p4q0_7d']

# Ensure no missing values
X_train_fe = X[features].fillna(0)
X_val_fe = new_X[features].fillna(0)

# Train XGBoost
model = xgb.XGBRegressor(n_estimators=380, learning_rate=0.01, max_depth=12, 
                         subsample=0.9, colsample_bytree=0.8, random_state=42)

model.fit(X_train_fe, y_train)

# Make predictions
y_pred = model.predict(X_val_fe)
y_pred = y_pred.round(0).astype(int)
y_pred = pd.DataFrame(y_pred)
y_pred.to_csv("y_pred_v6.csv")

In [ ]:
#Feature Engineering v2

# Copy dataset
X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
new_X = x_val[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
Y = y_train

# Feature Engineering: Encode "gare" variable
# Frequency encoding
gare_counts = x_train['gare'].value_counts()
X['gare_encoded'] = x_train['gare'].map(gare_counts)
new_X['gare_encoded'] = x_val['gare'].map(gare_counts).fillna(0)  # Fill unseen values with 0

# Train XGBoost Model
model = xgb.XGBRegressor(
    n_estimators=375,   # Number of trees (can be tuned)
    max_depth=12,        # Depth of each tree
    learning_rate=0.01,  # Step size (can be tuned)
    subsample=0.9,      # Row sampling
    colsample_bytree=0.8,  # Feature sampling
    random_state=42,
    n_jobs=-1           # Use all CPU cores
)

model.fit(X, Y)

# Make Predictions
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)

# Compute MAE
mae = mean_absolute_error(y_val, y_pred)
print(f"Local MAE after adding 'gare': {mae:.4f}")



Local MAE after adding 'gare': 0.7184


In [290]:
#Feature Engineering v3

# Copy dataset
X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
new_X = x_val[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
Y = y_train

# Feature Engineering: Encode "gare" variable
# Frequency encoding
gare_counts = x_train['gare'].value_counts()
X['gare_encoded'] = x_train['gare'].map(gare_counts)
new_X['gare_encoded'] = x_val['gare'].map(gare_counts).fillna(0)  # Fill unseen values with 0

arret_counts = x_train['arret'].value_counts()
X['arret_encoded'] = x_train['arret'].map(arret_counts)
new_X['arret_encoded'] = x_val['arret'].map(arret_counts).fillna(0)  # Fill unseen values with 0

# Train XGBoost Model
model = xgb.XGBRegressor(
    n_estimators=1250,   # Number of trees (can be tuned)
    max_depth=12,        # Depth of each tree
    learning_rate=0.01,  # Step size (can be tuned)
    subsample=0.9,      # Row sampling
    colsample_bytree=0.8,  # Feature sampling
    random_state=42,
    n_jobs=-1           # Use all CPU cores
)

model.fit(X, Y)

# Make Predictions
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)

# Compute MAE
mae = mean_absolute_error(y_val, y_pred)
print(f"Local MAE: {mae:.4f}")

Local MAE: 0.6939


In [ ]:
# Try 7: XGBoost Model with encoded "gare" and "arret" features (Score: 0,7023)

# Copy dataset
X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
new_X = x_outsample[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
Y = y_train

# Feature Engineering: Encode "gare" variable
# Frequency encoding
gare_counts = x_train['gare'].value_counts()
X['gare_encoded'] = x_train['gare'].map(gare_counts)
new_X['gare_encoded'] = x_outsample['gare'].map(gare_counts).fillna(0)  # Fill unseen values with 0

arret_counts = x_train['arret'].value_counts()
X['arret_encoded'] = x_train['arret'].map(arret_counts)
new_X['arret_encoded'] = x_outsample['arret'].map(arret_counts).fillna(0)  # Fill unseen values with 0

# Train XGBoost Model
model = xgb.XGBRegressor(
    n_estimators=1250,   # Number of trees (can be tuned)
    max_depth=12,        # Depth of each tree
    learning_rate=0.01,  # Step size (can be tuned)
    subsample=0.9,      # Row sampling
    colsample_bytree=0.8,  # Feature sampling
    random_state=42,
    n_jobs=-1           # Use all CPU cores
)

model.fit(X, Y)

# Make Predictions
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)
y_pred = pd.DataFrame(y_pred)
y_pred.to_csv("y_pred_v7.csv")

In [ ]:
#Feature Engineering v4

# Copy dataset
X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
new_X = x_val[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
Y = y_train

# Feature Engineering: Encode "gare" variable
gare_counts = x_train['gare'].value_counts()
X['gare_encoded'] = x_train['gare'].map(gare_counts)
new_X['gare_encoded'] = x_val['gare'].map(gare_counts).fillna(0)

arret_counts = x_train['arret'].value_counts()
X['arret_encoded'] = x_train['arret'].map(arret_counts)
new_X['arret_encoded'] = x_val['arret'].map(arret_counts).fillna(0)

# ➡️ **Rolling Statistics (Short-Term Trends)**
for window in [2, 8]:
    X[f'rolling_mean_{window}d'] = X[['p2q0', 'p3q0', 'p4q0']].mean(axis=1).rolling(window=window, min_periods=1).mean()
    X[f'rolling_std_{window}d'] = X[['p2q0', 'p3q0', 'p4q0']].std(axis=1).rolling(window=window, min_periods=1).std()

    new_X[f'rolling_mean_{window}d'] = new_X[['p2q0', 'p3q0', 'p4q0']].mean(axis=1).rolling(window=window, min_periods=1).mean()
    new_X[f'rolling_std_{window}d'] = new_X[['p2q0', 'p3q0', 'p4q0']].std(axis=1).rolling(window=window, min_periods=1).std()

# Ensure validation set has same columns
new_X = new_X.reindex(columns=X.columns, fill_value=0)

# Train XGBoost Model with More Trees & Lower Learning Rate
model = xgb.XGBRegressor(
    n_estimators=1250,   # More trees for better learning
    max_depth=12,        # Slightly reduced depth
    learning_rate=0.01, # Lower learning rate
    subsample=0.9,      
    colsample_bytree=0.8,  # More feature sampling
    random_state=42,
    n_jobs=-1
)

model.fit(X, Y)

# Make Predictions
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)

# Compute MAE
mae = mean_absolute_error(y_val, y_pred)
print(f"Local MAE (Optimized): {mae:.4f}")

Local MAE (Optimized): 0.6877


In [ ]:
# Try 8: XGBoost Model with Rolling Statistics feature (Score: 0,7130)

# Copy dataset
X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
new_X = x_outsample[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
Y = y_train

# Feature Engineering: Encode "gare" variable
# Frequency encoding
gare_counts = x_train['gare'].value_counts()
X['gare_encoded'] = x_train['gare'].map(gare_counts)
new_X['gare_encoded'] = x_outsample['gare'].map(gare_counts).fillna(0)  # Fill unseen values with 0

arret_counts = x_train['arret'].value_counts()
X['arret_encoded'] = x_train['arret'].map(arret_counts)
new_X['arret_encoded'] = x_outsample['arret'].map(arret_counts).fillna(0)  # Fill unseen values with 0

# ➡️ **Rolling Statistics (Short-Term Trends)**
for window in [2, 8]:
    X[f'rolling_mean_{window}d'] = X[['p2q0', 'p3q0', 'p4q0']].mean(axis=1).rolling(window=window, min_periods=1).mean()
    X[f'rolling_std_{window}d'] = X[['p2q0', 'p3q0', 'p4q0']].std(axis=1).rolling(window=window, min_periods=1).std()

    new_X[f'rolling_mean_{window}d'] = new_X[['p2q0', 'p3q0', 'p4q0']].mean(axis=1).rolling(window=window, min_periods=1).mean()
    new_X[f'rolling_std_{window}d'] = new_X[['p2q0', 'p3q0', 'p4q0']].std(axis=1).rolling(window=window, min_periods=1).std()

# Ensure validation set has same columns
new_X = new_X.reindex(columns=X.columns, fill_value=0)

# Train XGBoost Model
model = xgb.XGBRegressor(
    n_estimators=1250,   # Number of trees (can be tuned)
    max_depth=12,        # Depth of each tree
    learning_rate=0.01,  # Step size (can be tuned)
    subsample=0.9,      # Row sampling
    colsample_bytree=0.8,  # Feature sampling
    random_state=42,
    n_jobs=-1           # Use all CPU cores
)

model.fit(X, Y)

# Make Predictions
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)
y_pred = pd.DataFrame(y_pred)
y_pred.to_csv("y_pred_v8.csv")


In [ ]:
# Feature Engineering v5

# Copy dataset
X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
new_X = x_val[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
Y = y_train.copy()

# Feature Engineering: Encode "gare" variable
gare_counts = x_train['gare'].value_counts()
X['gare_encoded'] = x_train['gare'].map(gare_counts)
new_X['gare_encoded'] = x_val['gare'].map(gare_counts).fillna(0)

arret_counts = x_train['arret'].value_counts()
X['arret_encoded'] = x_train['arret'].map(arret_counts)
new_X['arret_encoded'] = x_val['arret'].map(arret_counts).fillna(0)

# Outlier detection (removing the 0.5% most extreme values)
def remove_outliers(df, target, threshold=0.012):
    """Removes the most extreme outliers in the target variable based on percentile."""
    lower_bound = Y.quantile(threshold)
    upper_bound = Y.quantile(1 - threshold)
    
    mask = (Y >= lower_bound) & (Y <= upper_bound)
    return df[mask], Y[mask]

X_filtered, Y_filtered = remove_outliers(X, Y, threshold=0.012)

# Ensure validation set has same columns
new_X = new_X.reindex(columns=X.columns, fill_value=0)

# Train XGBoost Model with More Trees & Lower Learning Rate
model = xgb.XGBRegressor(
    n_estimators=1400,   # More trees for better learning
    max_depth=12,        # Slightly reduced depth
    learning_rate=0.0095,  # Lower learning rate
    subsample=0.9,      
    colsample_bytree=0.8,  # More feature sampling
    random_state=54,
    n_jobs=-1
)

# Train the model on the filtered dataset
model.fit(X_filtered, Y_filtered)

# Make Predictions
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)

# Compute MAE
mae = mean_absolute_error(y_val, y_pred)
print(f"Local MAE : {mae:.4f}")

Local MAE : 0.6631


In [ ]:
# Try 9: XGBoost Model with Outlier detection (Score: 0,6619)

# Copy dataset
X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
new_X = x_outsample[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
Y = y_train

# Feature Engineering: Encode "gare" variable
# Frequency encoding
gare_counts = x_train['gare'].value_counts()
X['gare_encoded'] = x_train['gare'].map(gare_counts)
new_X['gare_encoded'] = x_outsample['gare'].map(gare_counts).fillna(0)  # Fill unseen values with 0

arret_counts = x_train['arret'].value_counts()
X['arret_encoded'] = x_train['arret'].map(arret_counts)
new_X['arret_encoded'] = x_outsample['arret'].map(arret_counts).fillna(0)  # Fill unseen values with 0

# Outlier detection (removing the 0.5% most extreme values)
def remove_outliers(df, target, threshold=0.012):
    """Removes the most extreme outliers in the target variable based on percentile."""
    lower_bound = Y.quantile(threshold)
    upper_bound = Y.quantile(1 - threshold)
    
    mask = (Y >= lower_bound) & (Y <= upper_bound)
    return df[mask], Y[mask]

X_filtered, Y_filtered = remove_outliers(X, Y, threshold=0.012)

# Ensure validation set has same columns
new_X = new_X.reindex(columns=X.columns, fill_value=0)

# Train XGBoost Model
model = xgb.XGBRegressor(
    n_estimators=1250,   # Number of trees (can be tuned)
    max_depth=12,        # Depth of each tree
    learning_rate=0.0095,  # Step size (can be tuned)
    subsample=0.9,      # Row sampling
    colsample_bytree=0.8,  # Feature sampling
    random_state=54,
    n_jobs=-1           # Use all CPU cores
)

model.fit(X_filtered, Y_filtered)

# Make Predictions
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)
y_pred = pd.DataFrame(y_pred)
y_pred.to_csv("y_pred_v9.csv")

In [ ]:
# Feature Engineering v6

# Copy dataset
X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
new_X = x_val[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
Y = y_train.copy()

# Feature Engineering: Encode "gare" variable
gare_counts = x_train['gare'].value_counts()
X['gare_encoded'] = x_train['gare'].map(gare_counts)
new_X['gare_encoded'] = x_val['gare'].map(gare_counts).fillna(0)

arret_counts = x_train['arret'].value_counts()
X['arret_encoded'] = x_train['arret'].map(arret_counts)
new_X['arret_encoded'] = x_val['arret'].map(arret_counts).fillna(0)

# ➡️ **Rolling Statistics (Short-Term Trends)**
for window in [2, 8]:
    X[f'rolling_mean_{window}d'] = X[['p2q0', 'p3q0', 'p4q0']].mean(axis=1).rolling(window=window, min_periods=1).mean()
    X[f'rolling_std_{window}d'] = X[['p2q0', 'p3q0', 'p4q0']].std(axis=1).rolling(window=window, min_periods=1).std()

    new_X[f'rolling_mean_{window}d'] = new_X[['p2q0', 'p3q0', 'p4q0']].mean(axis=1).rolling(window=window, min_periods=1).mean()
    new_X[f'rolling_std_{window}d'] = new_X[['p2q0', 'p3q0', 'p4q0']].std(axis=1).rolling(window=window, min_periods=1).std()

# Outlier detection (removing the 0.5% most extreme values)
def remove_outliers(df, target, threshold=0.012):
    """Removes the most extreme outliers in the target variable based on percentile."""
    lower_bound = Y.quantile(threshold)
    upper_bound = Y.quantile(1 - threshold)
    
    mask = (Y >= lower_bound) & (Y <= upper_bound)
    return df[mask], Y[mask]

X_filtered, Y_filtered = remove_outliers(X, Y, threshold=0.012)

# Ensure validation set has same columns
new_X = new_X.reindex(columns=X.columns, fill_value=0)

# Train XGBoost Model with More Trees & Lower Learning Rate
model = xgb.XGBRegressor(
    n_estimators=1400,   # More trees for better learning
    max_depth=12,        # Slightly reduced depth
    learning_rate=0.01,  # Lower learning rate
    subsample=0.9,      
    colsample_bytree=0.8,  # More feature sampling
    random_state=42,
    n_jobs=-1
)

# Train the model on the filtered dataset
model.fit(X_filtered, Y_filtered)

# Make Predictions
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)

# Compute MAE
mae = mean_absolute_error(y_val, y_pred)
print(f"Local MAE : {mae:.4f}")


Local MAE : 0.6556


In [ ]:
# Try 10: XGBoost Model with Outlier detection & rolling stat (Score: 0,6777)

# Copy dataset
X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
new_X = x_outsample[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
Y = y_train

# Feature Engineering: Encode "gare" variable
# Frequency encoding
gare_counts = x_train['gare'].value_counts()
X['gare_encoded'] = x_train['gare'].map(gare_counts)
new_X['gare_encoded'] = x_outsample['gare'].map(gare_counts).fillna(0)  # Fill unseen values with 0

arret_counts = x_train['arret'].value_counts()
X['arret_encoded'] = x_train['arret'].map(arret_counts)
new_X['arret_encoded'] = x_outsample['arret'].map(arret_counts).fillna(0)  # Fill unseen values with 0

# ➡️ **Rolling Statistics (Short-Term Trends)**
for window in [2, 8]:
    X[f'rolling_mean_{window}d'] = X[['p2q0', 'p3q0', 'p4q0']].mean(axis=1).rolling(window=window, min_periods=1).mean()
    X[f'rolling_std_{window}d'] = X[['p2q0', 'p3q0', 'p4q0']].std(axis=1).rolling(window=window, min_periods=1).std()

    new_X[f'rolling_mean_{window}d'] = new_X[['p2q0', 'p3q0', 'p4q0']].mean(axis=1).rolling(window=window, min_periods=1).mean()
    new_X[f'rolling_std_{window}d'] = new_X[['p2q0', 'p3q0', 'p4q0']].std(axis=1).rolling(window=window, min_periods=1).std()

# Outlier detection (removing the 0.5% most extreme values)
def remove_outliers(df, target, threshold=0.012):
    """Removes the most extreme outliers in the target variable based on percentile."""
    lower_bound = Y.quantile(threshold)
    upper_bound = Y.quantile(1 - threshold)
    
    mask = (Y >= lower_bound) & (Y <= upper_bound)
    return df[mask], Y[mask]

X_filtered, Y_filtered = remove_outliers(X, Y, threshold=0.012)

# Ensure validation set has same columns
new_X = new_X.reindex(columns=X.columns, fill_value=0)

# Train XGBoost Model
model = xgb.XGBRegressor(
    n_estimators=1250,   # Number of trees (can be tuned)
    max_depth=12,        # Depth of each tree
    learning_rate=0.01,  # Step size (can be tuned)
    subsample=0.9,      # Row sampling
    colsample_bytree=0.8,  # Feature sampling
    random_state=42,
    n_jobs=-1           # Use all CPU cores
)

model.fit(X_filtered, Y_filtered)

# Make Predictions
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)
y_pred = pd.DataFrame(y_pred)
y_pred.to_csv("y_pred_v10.csv")

In [358]:
# Feature Engineering v5

# Copy dataset
X = x_train[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
new_X = x_val[['p2q0', 'p3q0', 'p4q0', 'p0q2', 'p0q3', 'p0q4']].copy()
Y = y_train.copy()

# Feature Engineering: Encode "gare" variable
gare_counts = x_train['gare'].value_counts()
X['gare_encoded'] = x_train['gare'].map(gare_counts)
new_X['gare_encoded'] = x_val['gare'].map(gare_counts).fillna(0)

arret_counts = x_train['arret'].value_counts()
X['arret_encoded'] = x_train['arret'].map(arret_counts)
new_X['arret_encoded'] = x_val['arret'].map(arret_counts).fillna(0)

# Outlier detection (removing the 0.5% most extreme values)
def remove_outliers(df, target, threshold=0.005):
    """Removes the most extreme outliers in the target variable based on percentile."""
    lower_bound = target.quantile(threshold)
    upper_bound = target.quantile(1 - threshold)
    
    mask = (target >= lower_bound) & (target <= upper_bound)
    return df[mask], target[mask]

X_filtered, Y_filtered = remove_outliers(X, Y, threshold=0.005)

# Ensure validation set has same columns
new_X = new_X.reindex(columns=X.columns, fill_value=0)

# Train LightGBM Model
model = lgb.LGBMRegressor(
    n_estimators=5400,  # More trees for better learning
    max_depth=12,       # Slightly reduced depth
    learning_rate=0.01, # Lower learning rate
    subsample=0.9,      
    colsample_bytree=0.8,  # More feature sampling
    random_state=42,
    n_jobs=-1
)

# Train the model on the filtered dataset
model.fit(X_filtered, Y_filtered)

# Make Predictions
y_pred = model.predict(new_X)
y_pred = y_pred.round(0).astype(int)

# Compute MAE
mae = mean_absolute_error(y_val, y_pred)
print(f"Local MAE (LightGBM): {mae:.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.044099 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 497
[LightGBM] [Info] Number of data points in the train set: 662158, number of used features: 8
[LightGBM] [Info] Start training from score -0.097172


KeyboardInterrupt: 